# 📊 Unemployment Analysis Using Python

---

| Field | Detail |
|:---|:---|
| **Internship Company** | Horizon TechX |
| **Task** | Task 2 – Unemployment Analysis Using Python |
| **Student** | Tharani Natarajan |
| **College** | IFET College of Engineering |
| **Department** | Artificial Intelligence and Data Science |
| **Project Type** | Data Analysis / Exploratory Data Analysis (EDA) |
| **Date** | September 2026 |

---


## 2. Objective

This notebook presents a complete Exploratory Data Analysis (EDA) of unemployment trends across Indian states and union territories using the publicly available CMIE Unemployment in India Dataset (May 2019 – June 2020).

**Key objectives:**
1. Understand overall national unemployment trends across the observed 14-month period.
2. Identify and compare regional and state-level disparities in unemployment.
3. Analyze the Labour Participation Rate and Employment Level trends over time.
4. Evaluate measurable changes in unemployment during the COVID-19 lockdown period (March–June 2020) compared to the pre-lockdown baseline.
5. Quantify statistical relationships between unemployment, employment, and labour participation rates.
6. Generate evidence-based insights that are grounded exclusively in the actual data.


## 3. Dataset Information

| Attribute | Detail |
|:---|:---|
| **Dataset Name** | Unemployment in India Dataset |
| **Primary Source** | Centre for Monitoring Indian Economy (CMIE) |
| **Public Archive** | [GitHub – adduadnanali/Unemployment-Analysis-in-India](https://raw.githubusercontent.com/adduadnanali/Unemployment-Analysis-in-India/main/Unemployment%20in%20India.csv) |
| **File Name** | `Unemployment in India.csv` |
| **Frequency** | Monthly |
| **Time Period** | May 31, 2019 – June 30, 2020 (14 months) |
| **Geographic Scope** | 28 Indian States & Union Territories |
| **Area Classification** | Rural and Urban |
| **Raw Dimensions** | 768 rows × 7 columns |
| **Valid Observations** | 740 rows (after removing 28 trailing blank rows) |

**Disclaimer:** This dataset was sourced from a reputable public repository for academic analysis.  
It has **not** been supplied by Horizon TechX. All findings are descriptive and based solely on this dataset.


## 4. Import Libraries

We use industry-standard Python data science libraries:
- **pandas** – Data manipulation and analysis
- **numpy** – Numerical computing
- **matplotlib** – Low-level chart rendering and customization
- **seaborn** – Statistical visualizations
- **pathlib** – OS-independent path management
- **warnings** – Suppress harmless display warnings


In [ ]:
# Standard library
import pathlib
import warnings
warnings.filterwarnings("ignore")

# Data Science core
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid", font="sans-serif", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

# Resolved paths using pathlib
NOTEBOOK_DIR = pathlib.Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # notebooks/ → project root
DATA_FILE    = PROJECT_ROOT / "data" / "Unemployment in India.csv"
FIGURES_DIR  = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR  = PROJECT_ROOT / "outputs" / "reports"

print("Project Root:", PROJECT_ROOT)
print("Data File   :", DATA_FILE)
print("Figures Dir :", FIGURES_DIR)
print("Reports Dir :", REPORTS_DIR)


## 5. Load Dataset

We load the raw CSV file using `pandas.read_csv()` and immediately inspect its shape and column names.


In [ ]:
# Load raw dataset
df_raw = pd.read_csv(DATA_FILE)

print(f"Raw Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print("\nColumn Names (raw, before cleaning):")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i}] '{col}'")


In [ ]:
# First 5 rows
print("First 5 rows:")
df_raw.head()


In [ ]:
# Last 5 rows — notice trailing NaN rows
print("Last 5 rows:")
df_raw.tail()


## 6. Initial Data Inspection

Before cleaning, we thoroughly audit the raw dataset:
- Schema and data types
- Null / missing value counts per column
- Duplicate row count
- Sample statistics


In [ ]:
# Data types and null count overview
print("df.info() output:")
df_raw.info()


In [ ]:
# Missing values per column
print("Missing values per column (raw):")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_summary = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_summary)


In [ ]:
# Duplicate rows count
dups = df_raw.duplicated().sum()
print(f"Duplicate rows in raw data: {dups}")


In [ ]:
# Descriptive statistics (raw numeric columns only)
print("Descriptive statistics (raw data):")
df_raw.describe()


## 7. Data Cleaning

### Cleaning Steps Applied:
1. **Strip column header whitespace** – Raw headers contain leading spaces (e.g., `' Date'`, `' Frequency'`).
2. **Remove all-NaN trailing rows** – 28 entirely blank rows at end of file are dropped.
3. **Strip categorical string values** – `Region`, `Frequency`, and `Area` columns have inconsistent spacing.
4. **Parse date strings** – `Date` column is in `DD-MM-YYYY` format; converted to `datetime64[ns]`.
5. **Engineer temporal features** – `Year`, `Month`, `Month_Name`, `YearMonth`.
6. **Classify COVID period** – Pre-COVID (before 01-Mar-2020) vs COVID Period (March–June 2020).
7. **Validate numeric types** – All rate/count columns confirmed as `float64`.
8. **Sort dataset** – Chronologically and by region for consistent sequential analysis.


In [ ]:
# ── Step 1: Strip column name whitespace ──
df = df_raw.copy()
df.columns = df.columns.str.strip()
print("Cleaned Column Names:", df.columns.tolist())


In [ ]:
# ── Step 2: Drop completely blank rows (trailing NaN rows) ──
original_len = len(df)
df = df.dropna(how="all").reset_index(drop=True)
print(f"Rows before removal : {original_len}")
print(f"Rows after removal  : {len(df)}")
print(f"Blank rows removed  : {original_len - len(df)}")


In [ ]:
# ── Step 3: Strip whitespace from string columns ──
for col in ["Region", "Frequency", "Area"]:
    df[col] = df[col].astype(str).str.strip()

print("Unique 'Frequency' values:", df["Frequency"].unique())
print("Unique 'Area' values:", df["Area"].unique())
print("Unique Regions count:", df["Region"].nunique())


In [ ]:
# ── Step 4: Parse Date string → datetime ──
df["Date"] = pd.to_datetime(df["Date"].str.strip(), format="%d-%m-%Y")
print("Date column dtype:", df["Date"].dtype)
print("Date range:", df["Date"].min().strftime("%d %B %Y"), "to", df["Date"].max().strftime("%d %B %Y"))


In [ ]:
# ── Step 5: Feature engineering (temporal fields) ──
df["Year"]       = df["Date"].dt.year
df["Month"]      = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")
df["YearMonth"]  = df["Date"].dt.to_period("M")

# ── Step 6: COVID-19 period classification ──
# India announced national lockdown on March 24, 2020.
# Pre-COVID  : May 2019 – Feb 2020 (10 months baseline)
# COVID Period: March 2020 – June 2020 (4 months lockdown impact)
df["Period"] = np.where(df["Date"] < pd.Timestamp("2020-03-01"), "Pre-COVID", "COVID Period")

print("Period distribution:")
print(df["Period"].value_counts())


In [ ]:
# ── Step 7: Confirm numeric types ──
num_cols = [
    "Estimated Unemployment Rate (%)",
    "Estimated Employed",
    "Estimated Labour Participation Rate (%)"
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    
# ── Step 8: Sort ──
df = df.sort_values(by=["Date", "Region", "Area"]).reset_index(drop=True)

print("\nCleaned dataset shape:", df.shape)
print("\nCleaned column dtypes:")
print(df.dtypes)


In [ ]:
# Confirm zero nulls after cleaning
print("Missing values after cleaning:")
print(df[num_cols + ["Region", "Date", "Area"]].isnull().sum())

print("\nDuplicate rows after cleaning:", df.duplicated().sum())


In [ ]:
# Preview cleaned dataset
df.head(8)


## 8. Data Quality Analysis

A comprehensive data quality audit summarizing all dimensions of the cleaned dataset.


In [ ]:
# ── Data Quality Summary ──
unemp = df["Estimated Unemployment Rate (%)"]
employed = df["Estimated Employed"]
labour = df["Estimated Labour Participation Rate (%)"]

print("=" * 65)
print("  DATA QUALITY REPORT - UNEMPLOYMENT IN INDIA DATASET")
print("=" * 65)
print(f"  Total Valid Observations     : {len(df):,}")
print(f"  Columns (cleaned)            : {df.shape[1]}")
print(f"  Date Range                   : {df['Date'].min().strftime('%d %b %Y')} to {df['Date'].max().strftime('%d %b %Y')}")
print(f"  Months Covered               : {df['Date'].dt.to_period('M').nunique()} months")
print(f"  States / UTs                 : {df['Region'].nunique()}")
print(f"  Area Types                   : {', '.join(df['Area'].unique())}")
print(f"  Missing Values               : 0 in all key columns")
print(f"  Duplicate Rows               : {df.duplicated().sum()}")
print()
print("  UNEMPLOYMENT RATE STATISTICS:")
print(f"    Min                        : {unemp.min():.2f}%")
print(f"    Q1 (25th pct)              : {unemp.quantile(0.25):.2f}%")
print(f"    Median                     : {unemp.median():.2f}%")
print(f"    Mean                       : {unemp.mean():.2f}%")
print(f"    Q3 (75th pct)              : {unemp.quantile(0.75):.2f}%")
print(f"    Max                        : {unemp.max():.2f}%")
print(f"    Std Deviation              : {unemp.std():.2f}%")
print()
print("  LABOUR PARTICIPATION RATE:")
print(f"    Mean                       : {labour.mean():.2f}%")
print(f"    Range                      : {labour.min():.2f}% to {labour.max():.2f}%")
print()
print("  EMPLOYMENT ESTIMATES:")
print(f"    Mean Employed              : {employed.mean():,.0f} persons")
print(f"    Total Records Employed Sum : {employed.sum():,.0f} persons")
print("=" * 65)


## 9. Descriptive Statistics

Summary statistics broken down by Area (Rural vs Urban) and Period (Pre-COVID vs COVID Period).


In [ ]:
# Overall descriptive statistics
print("Overall descriptive statistics:")
df[num_cols].describe()


In [ ]:
# By Area: Rural vs Urban
print("Descriptive statistics by Area:")
df.groupby("Area")[num_cols].describe().round(2)


In [ ]:
# By Period: Pre-COVID vs COVID
print("Descriptive statistics by Period:")
df.groupby("Period")[num_cols].describe().round(2)


## 10. Overall Unemployment Trend Over Time

We aggregate national monthly unemployment means to visualize the 14-month trajectory, with the COVID-19 lockdown phase shaded for reference.


In [ ]:
# Monthly national mean unemployment
monthly = df.groupby("Date")["Estimated Unemployment Rate (%)"].mean().reset_index()
monthly.columns = ["Date", "Mean_Unemployment"]

print(f"Monthly national unemployment statistics:")
print(f"  Lowest  : {monthly['Mean_Unemployment'].min():.2f}% ({monthly.loc[monthly['Mean_Unemployment'].idxmin(), 'Date'].strftime('%b %Y')})")
print(f"  Highest : {monthly['Mean_Unemployment'].max():.2f}% ({monthly.loc[monthly['Mean_Unemployment'].idxmax(), 'Date'].strftime('%b %Y')})")
print(f"  Overall : {monthly['Mean_Unemployment'].mean():.2f}%")
print()
print(monthly.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(monthly["Date"], monthly["Mean_Unemployment"],
        color="#1f77b4", marker="o", linewidth=2.5, markersize=7,
        label="National Monthly Mean (%)")
ax.axhline(monthly["Mean_Unemployment"].mean(), color="gray", linestyle="--",
           linewidth=1.3, label=f"Period Avg ({monthly['Mean_Unemployment'].mean():.2f}%)")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-30"),
           color="#ff7f0e", alpha=0.18, label="COVID-19 Lockdown (Mar–Jun 2020)")

max_idx = monthly["Mean_Unemployment"].idxmax()
ax.annotate(
    f"Peak: {monthly.loc[max_idx, 'Mean_Unemployment']:.2f}%\n({monthly.loc[max_idx, 'Date'].strftime('%B %Y')})",
    xy=(monthly.loc[max_idx, "Date"], monthly.loc[max_idx, "Mean_Unemployment"]),
    xytext=(monthly.loc[max_idx, "Date"] - pd.Timedelta(days=80), monthly.loc[max_idx, "Mean_Unemployment"] + 3),
    arrowprops=dict(facecolor="#d62728", shrink=0.1, width=1.5, headwidth=8),
    fontsize=10, fontweight="bold", color="#d62728"
)

ax.set_title("Overall Unemployment Rate Trend in India (May 2019 – June 2020)",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Observation Month", fontsize=12, fontweight="bold")
ax.set_ylabel("Estimated Unemployment Rate (%)", fontsize=12, fontweight="bold")
plt.xticks(monthly["Date"], [d.strftime("%b %Y") for d in monthly["Date"]], rotation=45, ha="right")
ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="none")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_unemployment_trend_over_time.png")
plt.show()
print("Figure 1 saved.")


## 11. Regional Analysis

We compute descriptive statistics for each of the 28 states/UTs and rank them by mean unemployment rate. This is **descriptive analysis only** — ranking reflects estimated unemployment averages and does not imply economic judgment about any particular region.


In [ ]:
# Regional mean unemployment statistics
regional = df.groupby("Region")["Estimated Unemployment Rate (%)"].agg(
    Mean="mean", Median="median", Std="std", Min="min", Max="max"
).round(2).sort_values("Mean", ascending=False).reset_index()
regional.index = regional.index + 1
regional.index.name = "Rank"

print("Regional Unemployment Rankings (by Mean Rate):")
print(regional.to_string())


In [ ]:
# Top 5 highest and lowest
print("TOP 5 HIGHEST AVERAGE UNEMPLOYMENT REGIONS:")
print(regional.head(5)[["Region", "Mean", "Max"]].to_string())
print()
print("TOP 5 LOWEST AVERAGE UNEMPLOYMENT REGIONS:")
print(regional.tail(5)[["Region", "Mean", "Min"]].to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))
region_means = df.groupby("Region")["Estimated Unemployment Rate (%)"].mean().sort_values(ascending=True)

norm = plt.Normalize(region_means.min(), region_means.max())
colors = plt.cm.RdYlGn_r(norm(region_means.values))

bars = ax.barh(region_means.index, region_means.values, color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(df["Estimated Unemployment Rate (%)"].mean(), color="black", linestyle="--", linewidth=1.3,
           label=f"National Avg ({df['Estimated Unemployment Rate (%)'].mean():.2f}%)")

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.25, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}%", va="center", fontsize=8, color="#333")

ax.set_title("Average Unemployment Rate by State / Union Territory", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Average Estimated Unemployment Rate (%)", fontsize=12, fontweight="bold")
ax.set_ylabel("State / Region", fontsize=12, fontweight="bold")
ax.set_xlim(0, region_means.max() + 5)
ax.legend(loc="lower right", frameon=True, facecolor="white")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_avg_unemployment_by_region.png")
plt.show()
print("Figure 2 saved.")


## 12. Unemployment Distribution

The histogram + KDE shows the overall distribution shape of all unemployment rate observations.  
The right-skewed distribution indicates that most observations cluster at low rates, while a few high-rate observations (particularly during lockdown) pull the mean upward.


In [ ]:
rates = df["Estimated Unemployment Rate (%)"]

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(rates, kde=True, bins=25, color="#2ca02c", edgecolor="black", alpha=0.6, ax=ax)

ax.axvline(rates.mean(),        color="#d62728", linestyle="-",  linewidth=2, label=f"Mean: {rates.mean():.2f}%")
ax.axvline(rates.median(),      color="#1f77b4", linestyle="--", linewidth=2, label=f"Median: {rates.median():.2f}%")
ax.axvline(rates.quantile(.75), color="#ff7f0e", linestyle=":",  linewidth=2, label=f"Q3: {rates.quantile(.75):.2f}%")

ax.set_title("Distribution of Unemployment Rates in India (All Observations)", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Estimated Unemployment Rate (%)", fontsize=12, fontweight="bold")
ax.set_ylabel("Frequency", fontsize=12, fontweight="bold")
ax.legend(frameon=True, facecolor="white")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_unemployment_distribution.png")
plt.show()
print("Figure 3 saved.")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
palette = {"Rural": "#8c564b", "Urban": "#17becf"}

sns.boxplot(data=df, x="Area", y="Estimated Unemployment Rate (%)",
            palette=palette, ax=ax, width=0.45, fliersize=4)
sns.stripplot(data=df, x="Area", y="Estimated Unemployment Rate (%)",
              color="black", alpha=0.25, jitter=0.2, size=4, ax=ax)

means = df.groupby("Area")["Estimated Unemployment Rate (%)"].mean()
areas = df["Area"].unique().tolist()
for i, area in enumerate(areas):
    ax.scatter(i, means[area], color="red", s=80, zorder=5)
    ax.text(i + 0.12, means[area], f"  Mean: {means[area]:.2f}%", color="red", fontweight="bold", fontsize=10)

ax.set_title("Unemployment Rate: Rural vs. Urban Sectors (Boxplot)", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Geographic Sector", fontsize=12, fontweight="bold")
ax.set_ylabel("Estimated Unemployment Rate (%)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_unemployment_boxplot_by_area.png")
plt.show()
print("Figure 4 saved.")


## 13. Employment Analysis

We track the aggregate estimated employed population over the 14-month period, split by Rural and Urban sectors.  
A sharp contraction in employment is visible during the COVID-19 lockdown months (April–May 2020), followed by a partial recovery in June 2020.


In [ ]:
emp_monthly = df.groupby(["Date", "Area"])["Estimated Employed"].sum().unstack() / 1e6
emp_total   = df.groupby("Date")["Estimated Employed"].sum() / 1e6

print("Monthly total employed (Millions):")
print(emp_total.to_frame("Total_Employed_M").to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(emp_total.index, emp_total.values, color="#2b5c8f", marker="s", linewidth=2.5, label="Total Employed (M)")
ax.plot(emp_monthly.index, emp_monthly["Rural"], color="#55a868", linestyle="--", marker="o", label="Rural Employed (M)")
ax.plot(emp_monthly.index, emp_monthly["Urban"], color="#c44e52", linestyle="--", marker="^", label="Urban Employed (M)")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-30"),
           color="#ff7f0e", alpha=0.15, label="COVID Lockdown Period")

ax.set_title("Estimated Employed Population Trend (Millions) — May 2019 to June 2020",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Observation Month", fontsize=12, fontweight="bold")
ax.set_ylabel("Employed (Millions)", fontsize=12, fontweight="bold")
ax.legend(loc="lower left", frameon=True, facecolor="white")
plt.xticks(emp_total.index, [d.strftime("%b %Y") for d in emp_total.index], rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_employment_trend_over_time.png")
plt.show()
print("Figure 5 saved.")


## 14. Labour Participation Rate Analysis

The Labour Participation Rate (LPR) measures the proportion of working-age population that is either employed or actively seeking work.  
Declining LPR during the lockdown suggests that many workers exited the labour force entirely — a discouraged-worker effect common during severe economic shocks.


In [ ]:
lpr_area    = df.groupby(["Date", "Area"])["Estimated Labour Participation Rate (%)"].mean().unstack()
lpr_overall = df.groupby("Date")["Estimated Labour Participation Rate (%)"].mean()

print("Monthly national LPR (%):")
print(lpr_overall.to_frame("National_LPR").to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(lpr_overall.index, lpr_overall.values, color="#4c72b0", marker="o", linewidth=2.5, label="Overall National LPR (%)")
ax.plot(lpr_area.index, lpr_area["Rural"], color="#55a868", linestyle=":", marker="d", label="Rural LPR (%)")
ax.plot(lpr_area.index, lpr_area["Urban"], color="#c44e52", linestyle=":", marker="v", label="Urban LPR (%)")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-30"),
           color="#ff7f0e", alpha=0.15, label="COVID Period")

ax.set_title("Estimated Labour Participation Rate (LPR) — May 2019 to June 2020",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Observation Month", fontsize=12, fontweight="bold")
ax.set_ylabel("Labour Participation Rate (%)", fontsize=12, fontweight="bold")
ax.legend(loc="upper right", frameon=True, facecolor="white")
plt.xticks(lpr_overall.index, [d.strftime("%b %Y") for d in lpr_overall.index], rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_labour_participation_trend.png")
plt.show()
print("Figure 6 saved.")


## 15. COVID-19 Impact Analysis

The dataset spans both sides of the Indian national lockdown announced on **March 24, 2020**.

**Period Definitions:**
- **Pre-COVID Baseline**: May 31, 2019 – February 29, 2020 (10 months, 536 observations)
- **COVID Lockdown Period**: March 31, 2020 – June 30, 2020 (4 months, 204 observations)

> ⚠️ **Note:** The observed increase in unemployment during the COVID period reflects a correlation with the lockdown timing. Multiple economic factors influence unemployment simultaneously. This analysis describes observed changes; it does not claim exclusive causation.


In [ ]:
# COVID-period breakdown
covid_stats = df.groupby("Period").agg(
    Observations   = ("Date", "count"),
    Mean_Unemp     = ("Estimated Unemployment Rate (%)", "mean"),
    Median_Unemp   = ("Estimated Unemployment Rate (%)", "median"),
    Max_Unemp      = ("Estimated Unemployment Rate (%)", "max"),
    Std_Unemp      = ("Estimated Unemployment Rate (%)", "std"),
    Mean_Employed  = ("Estimated Employed", "mean"),
    Mean_LPR       = ("Estimated Labour Participation Rate (%)", "mean")
).reindex(["Pre-COVID", "COVID Period"]).round(2)

print("COVID Period Comparison Summary:")
print(covid_stats.to_string())

pre_mean = covid_stats.loc["Pre-COVID", "Mean_Unemp"]
cov_mean = covid_stats.loc["COVID Period", "Mean_Unemp"]
abs_diff = cov_mean - pre_mean
pct_diff = (abs_diff / pre_mean) * 100

print(f"\nAbsolute Change in Mean Unemployment : +{abs_diff:.2f} percentage points")
print(f"Relative Change (%)                   : +{pct_diff:.2f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

top_regions = df.groupby("Region")["Estimated Employed"].mean().nlargest(12).index.tolist()
comp_df = df[df["Region"].isin(top_regions)].copy()

period_agg = comp_df.groupby(["Region", "Period"])["Estimated Unemployment Rate (%)"].mean().unstack()
overall_period = df.groupby("Period")["Estimated Unemployment Rate (%)"].mean()
period_agg.loc["National Average"] = overall_period
period_agg = period_agg.sort_values("COVID Period", ascending=True)

x = np.arange(len(period_agg))
width = 0.35
ax.barh(x - width/2, period_agg["Pre-COVID"],     width,
        label="Pre-COVID (May 2019–Feb 2020)",       color="#2ca02c", alpha=0.85, edgecolor="black", linewidth=0.4)
ax.barh(x + width/2, period_agg["COVID Period"],  width,
        label="COVID Period (Mar–Jun 2020)",         color="#d62728", alpha=0.85, edgecolor="black", linewidth=0.4)

ax.set_yticks(x)
ax.set_yticklabels(period_agg.index, fontsize=9.5)
ax.set_xlabel("Mean Estimated Unemployment Rate (%)", fontsize=12, fontweight="bold")
ax.set_title("Unemployment Rate: Pre-COVID vs COVID Lockdown Period", fontsize=14, fontweight="bold", pad=14)
ax.legend(loc="lower right", frameon=True, facecolor="white")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_covid19_period_comparison.png")
plt.show()
print("Figure 7 saved.")


## 16. Correlation Analysis

We compute the **Pearson correlation** matrix among the three numerical labor market indicators.

> ⚠️ **Correlation ≠ Causation.** The values below describe statistical co-movement between variables, not causal relationships. Economic forces, policy interventions, and structural factors operate simultaneously.


In [ ]:
numeric_cols = [
    "Estimated Unemployment Rate (%)",
    "Estimated Employed",
    "Estimated Labour Participation Rate (%)"
]

corr = df[numeric_cols].corr()
print("Pearson Correlation Matrix:")
print(corr.round(4))


In [ ]:
labels = ["Unemployment\nRate (%)", "Estimated\nEmployed", "Labour Participation\nRate (%)"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", cbar=True,
            xticklabels=labels, yticklabels=labels, vmin=-1, vmax=1,
            linewidths=1, linecolor="white", ax=ax,
            annot_kws={"size": 12, "weight": "bold"})

ax.set_title("Correlation Heatmap: Key Labor Market Indicators", fontsize=13, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_correlation_heatmap.png")
plt.show()
print("Figure 8 saved.")


## 17. Key Findings

All findings below are derived **exclusively from the actual dataset values** — no fabricated or assumed figures.


In [ ]:
unemp = df["Estimated Unemployment Rate (%)"]
regional = df.groupby("Region")["Estimated Unemployment Rate (%)"].mean().sort_values(ascending=False)

pre_covid  = df[df["Period"] == "Pre-COVID"]["Estimated Unemployment Rate (%)"].mean()
covid_mean = df[df["Period"] == "COVID Period"]["Estimated Unemployment Rate (%)"].mean()
rural_mean = df[df["Area"] == "Rural"]["Estimated Unemployment Rate (%)"].mean()
urban_mean = df[df["Area"] == "Urban"]["Estimated Unemployment Rate (%)"].mean()

print("=" * 65)
print("  KEY ANALYTICAL FINDINGS")
print("=" * 65)

print(f"\n  1. National Mean Unemployment Rate   : {unemp.mean():.2f}%")
print(f"     National Median Unemployment Rate  : {unemp.median():.2f}%")
print(f"     Minimum Observed Rate              : {unemp.min():.2f}%")
print(f"     Maximum Observed Rate              : {unemp.max():.2f}%")

print(f"\n  2. COVID-19 Period Comparison:")
print(f"     Pre-COVID Avg (May 2019–Feb 2020)  : {pre_covid:.2f}%")
print(f"     COVID Period Avg (Mar–Jun 2020)    : {covid_mean:.2f}%")
print(f"     Absolute Increase                  : +{covid_mean - pre_covid:.2f} percentage points")
print(f"     Relative Increase                  : +{((covid_mean - pre_covid) / pre_covid * 100):.2f}%")

print(f"\n  3. Area Comparison:")
print(f"     Rural Mean Unemployment            : {rural_mean:.2f}%")
print(f"     Urban Mean Unemployment            : {urban_mean:.2f}%")

print(f"\n  4. States with Highest Avg Unemployment:")
for rank, (region, rate) in enumerate(regional.head(3).items(), 1):
    print(f"     {rank}. {region}: {rate:.2f}%")

print(f"\n  5. States with Lowest Avg Unemployment:")
for rank, (region, rate) in enumerate(regional.tail(3).items(), 1):
    print(f"     {rank}. {region}: {rate:.2f}%")

corr = df[numeric_cols].corr()
print(f"\n  6. Correlation between Unemployment Rate and LPR: {corr.loc['Estimated Unemployment Rate (%)', 'Estimated Labour Participation Rate (%)']:.3f}")
print(f"     (Weak negative correlation — note: correlation does not imply causation)")
print("=" * 65)


## 18. Limitations

| Limitation | Description |
|:---|:---|
| **Dataset time range** | Observations end in June 2020. The second and third COVID-19 waves (2021), and post-pandemic recovery patterns, are not captured. |
| **Aggregate estimates** | The CMIE data provides state-level monthly estimates, not individual-level survey microdata. Interpretation must be at aggregate scale. |
| **No causal inference** | This is descriptive EDA. Establishing causal links between the lockdown and specific unemployment levels requires controlled econometric methods beyond the scope of this project. |
| **Geographic coverage** | 28 regions are represented; some smaller union territories may have higher uncertainty in their estimates. |
| **Frequency** | Monthly resolution does not capture intra-month disruptions (e.g., the day-of-lockdown shock on March 24, 2020). |
| **External factors** | Seasonal agricultural patterns, pre-existing state-level economic conditions, and fiscal policies independently affect unemployment and are not controlled in this dataset. |


## 19. Conclusion

This project delivers a complete, evidence-based Exploratory Data Analysis of unemployment in India across 14 months (May 2019 – June 2020) using the publicly available CMIE-sourced dataset.

**Summary of Findings:**

- The national mean unemployment rate was **11.79%** across all 740 observations, with a median of **8.35%** — indicating a positively skewed distribution where acute lockdown-period spikes drive the mean upward.
- The unemployment rate increased from an average of **9.51%** (Pre-COVID baseline) to **17.77%** during the COVID-19 lockdown period — an absolute increase of **+8.26 percentage points** (+86.91% relative increase).
- This surge coincided with a notable contraction in the estimated employed population and a decline in the labour participation rate during April–May 2020.
- Significant regional variation exists across India's 28 states and union territories, with some regions consistently exhibiting above-average unemployment rates throughout the observed period.
- Urban sectors recorded a slightly higher mean unemployment rate than rural sectors, potentially reflecting the greater sensitivity of formal and service-sector urban employment to mobility restrictions.
- The correlation between unemployment rate and labour participation rate is weak-to-negative, suggesting that labour force withdrawal partially masks the true extent of employment distress during the lockdown.

**This analysis is purely descriptive.** The patterns observed are consistent with the extraordinary economic disruption of the COVID-19 lockdown period, but all relationships documented here are correlational, not causal.

---
*Task 2 – Horizon TechX Internship | Tharani Natarajan | IFET College of Engineering*
